In [ ]:
import os
import tensorflow as tf

os.environ['CUDA_VISIBLE_DEVICES'] = '5'
gpus = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
tf.config.experimental.set_virtual_device_configuration(
    gpus[0],
    [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=1024)]
)

I0000 00:00:1775139278.731143  539468 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import os
import torch

print(os.getcwd())
BASE_PATH = "../"
device = "cuda" if torch.cuda.is_available() else "cpu"

HF_TOKEN = ...

/data/koposovti/lecture-transcriber/notebooks


In [3]:
import subprocess
import pandas as pd
from transformers import pipeline
import json
import time
import glob
from tqdm import tqdm

/data/koposovti/lecture-transcriber/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
chunk_duration = 300 # 300 secs = 5 mins btw

In [5]:
def split_audio(input_path, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        
    for f in glob.glob(os.path.join(output_directory, "*.mp3")):
        os.remove(f)

    cmd = [
        "ffmpeg", "-i", input_path, 
        "-f", "segment",
        "-segment_time", str(chunk_duration),
        "-c", "copy", os.path.join(output_directory, "%03d.mp3")
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    return sorted(glob.glob(os.path.join(output_directory, "*.mp3")))



In [23]:
from faster_whisper import WhisperModel

def transcribe_lecture(audio_path, save_path):
    model = WhisperModel("medium", device="cuda", compute_type="float16", use_auth_token=HF_TOKEN)
    
    segments, info = model.transcribe(audio_path, beam_size=5, language="ru", vad_filter=True)
    
    results = []
    for segment in segments:
        results.append({
            "start": segment.start,
            "end": segment.end,
            "text": segment.text
        })
    
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
        
    return pd.DataFrame(results)

In [24]:
import os
import glob
from tqdm import tqdm

RAW_DIR = BASE_PATH + "data/raw"
TRANSCRIPTIONS_BASE = BASE_PATH + "data/transcriptions"
subdirs = sorted([d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))])
print(f"Found {len(subdirs)} directories: {subdirs}")


Found 7 directories: ['00', '01', '02', '03', '04', '05', '06']


In [ ]:
subdir = subdirs[0] + "_"
input_path_dir = os.path.join(RAW_DIR, subdirs[0])
output_path_dir = os.path.join(TRANSCRIPTIONS_BASE, subdir)
output_chunks_dir = os.path.join(output_path_dir, "chunks")
save_json_path = os.path.join(output_path_dir, "lecture_raw.json")

os.makedirs(output_chunks_dir, exist_ok=True)

mp3_files = glob.glob(os.path.join(input_path_dir, "*.mp3"))

audio_file = mp3_files[0]

df_transcript = transcribe_lecture(
    audio_path=audio_file,
    save_path=save_json_path,
)

df_transcript.head()

,start,end,text
0,0.00,8.70,У вас должна отображаться презентация.
1,8.70,11.86,"Да, все видно."
2,11.86,12.86,Хорошо.
3,12.86,15.86,Давайте тогда.
4,15.86,18.38,Супер.


In [ ]:
for subdir in tqdm(subdirs, desc="Transcription"):
    input_path_dir = os.path.join(RAW_DIR, subdir)
    output_path_dir = os.path.join(TRANSCRIPTIONS_BASE, subdir)
    output_chunks_dir = os.path.join(output_path_dir, "chunks")
    save_json_path = os.path.join(output_path_dir, "lecture_raw.json")
    
    os.makedirs(output_chunks_dir, exist_ok=True)
    
    mp3_files = glob.glob(os.path.join(input_path_dir, "*.mp3"))
    if not mp3_files:
        print(f"Warning: no .mp3 found in {subdir}")
        continue
    if len(mp3_files) > 1:
        print(f"Warning: found multiple .mp3 files in {subdir}")
    
    audio_file = mp3_files[0]
    
    try:
        transcribe_lecture(
            audio_path=audio_file,
            save_path=save_json_path,
        )
    except Exception as e:
        print(f"Error during processing {subdir}: {e}")


Transcription: 100%|██████████| 7/7 [1:21:48<00:00, 701.26s/it]

Transcription completed!


: 